# Topic Naming with OpenAI

Loads the BERTopic models and their topic info produced in `02-topic_model/`,
sends representative documents per cluster to GPT to obtain:

1. A **description** of each topic cluster.
2. An **enhanced (synthesised) description**.
3. A short **cluster name**.

Prompts are loaded from `prompts.yaml`. Results are saved as TSV files alongside
the topic models in `assets/topic_models/`.

In [1]:
import os
import yaml
import numpy as np
import pandas as pd
from openai import OpenAI

In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')

# Number of representative documents to send per cluster
TOP_N_DOCS = 5

# OpenAI model
MODEL = 'gpt-4.1-nano'

# ── Load prompts ──────────────────────────────────────────────────────────────
with open('prompts.yaml') as f:
    prompts = yaml.safe_load(f)

THEME = prompts['theme']
THEME_DESCRIPTION = prompts['theme_description']

print(f'Theme: {THEME}')
print(f'Description: {THEME_DESCRIPTION[:80]}...')

Theme: Synthetic Biology
Description: Synthetic biology is an interdisciplinary field that combines principles from bi...


In [3]:
# ── OpenAI client ─────────────────────────────────────────────────────────────
api_key = open('openai.key').read().strip()
client = OpenAI(api_key=api_key)

## 1. Load topic models and corpora

In [4]:
# Topic-level info (from BERTopic's get_topic_info())
teams_info  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'),  sep='\t')
papers_info = pd.read_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t')

# Document-level topic assignments
teams_doc_topics  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_doc_topics.txt'),  sep='\t')
papers_doc_topics = pd.read_csv(os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t')

# Full corpora (with text)
teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'),  sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

# Merge text into doc-topics
teams_df  = teams_doc_topics.merge(teams_corpus,  on='UT', how='left')
papers_df = papers_doc_topics.merge(papers_corpus, on='id', how='left')

print(f'Teams:  {len(teams_df):,} docs, {teams_info.Topic.max() + 1} topics')
print(f'Papers: {len(papers_df):,} docs, {papers_info.Topic.max() + 1} topics')

Teams:  4,548 docs, 103 topics
Papers: 33,625 docs, 234 topics


## 2. Helper functions

In [5]:
def ask_gpt(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    temperature: float = 0.1,
    max_tokens: int = 500,
) -> str:
    """Send a system + user prompt to OpenAI and return the assistant reply."""
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    )
    return response.choices[0].message.content


def get_representative_texts(df: pd.DataFrame, topic_id: int, top_n: int = TOP_N_DOCS) -> str:
    """Return the top-N documents of a topic joined with ##### separators.

    The text is truncated to ~14 000 chars (~3 500 tokens) to stay within
    context limits.
    """
    subset = df[df['topic'] == topic_id]
    texts = subset['text'].head(top_n).tolist()
    bulk = ' ##### '.join(texts)
    return bulk[:14_000]


def fmt_prompt(template: str, **kwargs) -> str:
    """Format a prompt template with the given keyword arguments."""
    result = template
    for key, value in kwargs.items():
        result = result.replace('{' + key + '}', str(value))
    return result

In [6]:
def name_topics(
    df: pd.DataFrame,
    topic_info: pd.DataFrame,
) -> pd.DataFrame:
    """For each non-outlier topic, get description, enhanced description, and name.

    Returns a DataFrame with columns: topic, description, enhanced_description, name.
    """
    p_desc = prompts['cluster_description']
    p_enh  = prompts['cluster_description_enhanced']
    p_name = prompts['cluster_name']

    rows = []
    topic_ids = sorted(topic_info[topic_info['Topic'] != -1]['Topic'].unique())

    for tid in topic_ids:
        print(f'  Topic {tid} ...', end=' ', flush=True)
        cluster_text = get_representative_texts(df, tid)

        # Step 1: cluster description
        description = ask_gpt(
            system_prompt=fmt_prompt(p_desc['system'], theme=THEME, theme_description=THEME_DESCRIPTION),
            user_prompt=fmt_prompt(p_desc['user'], cluster_text=cluster_text),
            temperature=0.2,
        )

        # Step 2: enhanced (synthesised) description
        enhanced = ask_gpt(
            system_prompt=fmt_prompt(p_enh['system'], theme=THEME),
            user_prompt=fmt_prompt(p_enh['user'], cluster_description=description),
            temperature=0.1,
        )

        # Step 3: short name
        name = ask_gpt(
            system_prompt=fmt_prompt(p_name['system'], theme=THEME, theme_description=THEME_DESCRIPTION),
            user_prompt=fmt_prompt(p_name['user'], cluster_description=description),
            max_tokens=60,
            temperature=0.3,
        )
        name = name.strip().strip('"').strip("'")

        print(name)
        rows.append({
            'topic': tid,
            'name': name,
            'description': enhanced,
            'raw_description': description,
        })

    return pd.DataFrame(rows)

## 3. Name iGEM Teams topics

In [7]:
print('Naming iGEM Teams topics...')
teams_names = name_topics(teams_df, teams_info)
teams_names

Naming iGEM Teams topics...
  Topic 0 ... Environmental Bioremediation and Resource Recovery
  Topic 1 ... Synthetic Biology-Enabled Disease Diagnostics
  Topic 2 ... Light-Controlled Synthetic Biology Systems
  Topic 3 ... Synthetic Biology for Bacterial Quorum Sensing and Biofilm Control
  Topic 4 ... Synthetic Biology for Plastic Biodegradation
  Topic 5 ... Microbe-Mediated Cancer Therapy
  Topic 6 ... Gut Microbiome Synthetic Biology
  Topic 7 ... Portable Diagnostics
  Topic 8 ... Synthetic Biology-Driven Biosensing Technologies
  Topic 9 ... Temperature-Regulated Synthetic Biological Systems
  Topic 10 ... Synthetic Biology Diagnostics
  Topic 11 ... Synthetic Biology for Environmental Pollution Control and Sustainable Agriculture
  Topic 12 ... Synthetic Biology for Programmable Cellular Communication and Computation
  Topic 13 ... Synthetic Biology for Diabetes
  Topic 14 ... Synthetic Biology for Phosphorus Recycling
  Topic 15 ... Synthetic Biology for Genetic Engineering an

,topic,name,description,raw_description
0,0,Environmental Bioremediation and Resource Reco...,This cluster centers on harnessing synthetic b...,The main topic of this cluster is the applicat...
1,1,Synthetic Biology-Enabled Disease Diagnostics,This collection of texts focuses on advancing ...,The common topic of these texts is the develop...
2,2,Light-Controlled Synthetic Biology Systems,This cluster focuses on the advancement and ut...,The main topic of this cluster is the developm...
3,3,Synthetic Biology for Bacterial Quorum Sensing...,This cluster focuses on leveraging synthetic b...,The common topic of this cluster is the applic...
4,4,Synthetic Biology for Plastic Biodegradation,This cluster centers on leveraging synthetic b...,The common topic of these texts is the applica...
...,...,...,...,...
98,98,Synthetic Biology for Pesticide Detection and ...,This cluster centers on leveraging genetic eng...,The common topic across these texts is the use...
99,99,Synthetic Biology-Driven Antiviral Therapeutic...,This collection of texts focuses on harnessing...,The common topic of these texts is the applica...
100,100,Synthetic Biology for Biological Computing and...,This collection of texts centers on leveraging...,The common topic across these texts is the app...
101,101,Synthetic Biology for Advanced Biological Syst...,This cluster centers on the innovative use of ...,The common topic across these texts is the app...


## 4. Name SynBio Papers topics

In [8]:
print('Naming SynBio Papers topics...')
papers_names = name_topics(papers_df, papers_info)
papers_names

Naming SynBio Papers topics...
  Topic 0 ... Synthetic Vesicle-Based Artificial Cells
  Topic 1 ... Governance and Responsible Innovation in Synthetic Biology
  Topic 2 ... Plant-Derived Bioactive Compound Synthesis
  Topic 3 ... Plant Synthetic Regulatory Elements
  Topic 4 ... Synthetic DNA Motifs for Immune Activation
  Topic 5 ... DNA Biosensing and Diagnostics
  Topic 6 ... Synthetic Biology for Sustainable Biofuel and Biochemical Production
  Topic 7 ... Promoter Engineering in Synthetic Biology
  Topic 8 ... Synthetic Biology-Driven Therapeutic Engineering
  Topic 9 ... Neuroreceptor and Neuropeptide Gene Regulation
  Topic 10 ... CRISPR-Cas Technologies
  Topic 11 ... Synthetic Biological Oscillators and Rhythms
  Topic 12 ... Cell-Free Synthetic Biology
  Topic 13 ... Genetic and Molecular Engineering in Synthetic Biology
  Topic 14 ... Microbial Genome Engineering
  Topic 15 ... Synthetic Biology in Viral Vaccines
  Topic 16 ... RNA-Based Regulatory Circuits
  Topic 17 ... Sy

,topic,name,description,raw_description
0,0,Synthetic Vesicle-Based Artificial Cells,The focus of this cluster centers on the desig...,The common topic across these texts is the des...
1,1,Governance and Responsible Innovation in Synth...,The cluster focuses on the governance and resp...,The common topic across these texts is the gov...
2,2,Plant-Derived Bioactive Compound Synthesis,This cluster focuses on leveraging synthetic b...,The common topic of this cluster is the applic...
3,3,Plant Synthetic Regulatory Elements,This cluster focuses on the development and ap...,"The common topic of these texts is the design,..."
4,4,Synthetic DNA Motifs for Immune Activation,This cluster focuses on the strategic use of s...,The common topic across these texts is the use...
...,...,...,...,...
229,229,DNA Nanomachines,This cluster centers on the design and utiliza...,The common topic of these texts is the develop...
230,230,Synthetic Biology of Cannabinoids,This cluster centers on the innovative manipul...,The common topic across these texts is the **S...
231,231,In Vitro Synthetic Biology,This cluster centers on the creation of in vit...,The common topic across these texts is the dev...
232,232,Mitochondrial Protein Engineering,This collection of texts centers on utilizing ...,The common topic of these texts is the applica...


## 5. Save results

In [9]:
teams_names.to_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'), sep='\t', index=False)
papers_names.to_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t', index=False)

print(f'Saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')

Saved to ../assets/topic_models
  papers_doc_topics.txt
  papers_grid_search.txt
  papers_topic_info.txt
  papers_topic_model
  papers_topic_names.txt
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
